## API


###IMSWeatherAPI

In [4]:
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from typing import Optional, Dict
import requests
import json
from datetime import datetime
from typing import Optional, Dict, List

class IMSWeatherAPI:
    """
    Client for the Israeli Meteorological Service (IMS) Weather Data API
    """

    BASE_URL = "https://api.ims.gov.il/v1/envista"

    def __init__(self, api_token: str):
        """
        Initialize the API client with the API token

        Args:
            api_token: API token from IMS
        """
        self.api_token = api_token
        self.headers = {
            "Authorization": f"ApiToken {api_token}"
        }

    def get_all_stations(self) -> Dict:
        """Get metadata for all weather stations"""
        url = f"{self.BASE_URL}/stations"
        response = requests.get(url, headers=self.headers)
        response.raise_for_status()
        return response.json()

    def get_station_info(self, station_id: int) -> Dict:
        """
        Get metadata for a specific station

        Args:
            station_id: Station number
        """
        url = f"{self.BASE_URL}/stations/{station_id}"
        response = requests.get(url, headers=self.headers)
        response.raise_for_status()
        return response.json()

    def get_all_regions(self) -> Dict:
        """Get metadata for all regions (including stations in each region)"""
        url = f"{self.BASE_URL}/regions"
        response = requests.get(url, headers=self.headers)
        response.raise_for_status()
        return response.json()

    def get_region_info(self, region_id: int) -> Dict:
        """
        Get metadata for a specific region

        Args:
            region_id: Region number
        """
        url = f"{self.BASE_URL}/regions/{region_id}"
        response = requests.get(url, headers=self.headers)
        response.raise_for_status()
        return response.json()

    def get_hourly_data_by_date_range(self, station_id: int,
                                      from_year: int, from_month: int, from_day: int,
                                      to_year: int, to_month: int, to_day: int,
                                      channel_id: Optional[int] = None) -> Dict:
        """
        Get hourly data for a date range (filters 10-minute data to keep every 6th record)

        Args:
            station_id: Station number
            from_year, from_month, from_day: Start date
            to_year, to_month, to_day: End date
            channel_id: Optional channel number

        Returns:
            Dict with filtered data containing hourly intervals
        """
        # Get the full 10-minute data
        data = self.get_data_by_date_range(
            station_id, from_year, from_month, from_day,
            to_year, to_month, to_day, channel_id
        )

        # Filter to keep every 6th data point (hourly intervals)
        if 'data' in data and isinstance(data['data'], list):
            data['data'] = data['data'][::6]

        return data

    def get_data_by_date_range(self, station_id: int,
                              from_year: int, from_month: int, from_day: int,
                              to_year: int, to_month: int, to_day: int,
                              channel_id: Optional[int] = None,
                              batch_months: int = 6) -> Dict:
        """
        Get data for a date range, splitting into batches for large ranges

        Args:
            station_id: Station number
            from_year, from_month, from_day: Start date
            to_year, to_month, to_day: End date
            channel_id: Optional channel number
            batch_months: Number of months per batch (default: 6)
        """
        start_date = datetime(from_year, from_month, from_day)
        end_date = datetime(to_year, to_month, to_day)

        # Calculate the difference in months
        month_diff = (end_date.year - start_date.year) * 12 + (end_date.month - start_date.month)

        # If the range is within batch_months, make a single request
        if month_diff <= batch_months:
            print(f"Fetching data for station {station_id} from {start_date.date()} to {end_date.date()}...")

            from_date = f"{from_year}/{from_month:02d}/{from_day:02d}"
            to_date = f"{to_year}/{to_month:02d}/{to_day:02d}"

            if channel_id:
                url = f"{self.BASE_URL}/stations/{station_id}/data/{channel_id}?from={from_date}&to={to_date}"
            else:
                url = f"{self.BASE_URL}/stations/{station_id}/data?from={from_date}&to={to_date}"

            response = requests.get(url, headers=self.headers)
            response.raise_for_status()
            print("✓ Data fetched successfully")
            return response.json()

        # Calculate total number of batches
        total_batches = (month_diff // batch_months) + (1 if month_diff % batch_months > 0 else 0)
        print(f"\n{'='*60}")
        print(f"Fetching data for station {station_id}")
        print(f"Date range: {start_date.date()} to {end_date.date()}")
        print(f"Total batches: {total_batches} (each ~{batch_months} months)")
        print(f"{'='*60}\n")

        # Split into batches
        all_data = None
        current_date = start_date
        batch_num = 0

        while current_date < end_date:
            batch_num += 1

            # Calculate batch end date (6 months from current_date or end_date, whichever is earlier)
            batch_end = min(current_date + relativedelta(months=batch_months), end_date)

            # Format dates for API
            from_date = current_date.strftime("%Y/%m/%d")
            to_date = batch_end.strftime("%Y/%m/%d")

            # Progress indicator
            print(f"[{batch_num}/{total_batches}] Fetching: {current_date.date()} → {batch_end.date()}", end=" ... ")

            # Make API request for this batch
            if channel_id:
                url = f"{self.BASE_URL}/stations/{station_id}/data/{channel_id}?from={from_date}&to={to_date}"
            else:
                url = f"{self.BASE_URL}/stations/{station_id}/data?from={from_date}&to={to_date}"

            response = requests.get(url, headers=self.headers)
            response.raise_for_status()
            batch_data = response.json()

            print("✓")

            # Merge results
            if all_data is None:
                all_data = batch_data
            else:
                if isinstance(batch_data, dict) and 'data' in batch_data:
                    all_data['data'].extend(batch_data['data'])
                elif isinstance(batch_data, list):
                    all_data.extend(batch_data)
                else:
                    all_data = {**all_data, **batch_data}

            # Move to next batch
            current_date = batch_end

        print(f"\n{'='*60}")
        print(f"✓ All batches completed successfully!")
        print(f"{'='*60}\n")

        return all_data

### Gap filler for data

In [5]:
import pandas as pd
from datetime import datetime

def fill_data_gaps(data: List[Dict], max_gap_hours: int = 6) -> pd.DataFrame:
    """
    Fill gaps in weather data using time-based interpolation.
    """
    df = pd.DataFrame(data)
    df['date_time'] = pd.to_datetime(df['date_time'], utc=True)
    df.set_index('date_time', inplace=True)
    df.sort_index(inplace=True)

    numeric_cols = ['temperature', 'humidity', 'solar_radiation', 'pressure']

    for col in numeric_cols:
        if col not in df.columns:
            continue

        # Step 1: Interpolate small gaps
        df[col] = df[col].interpolate(method='time', limit=max_gap_hours)

        # Step 2: Fill remaining with hourly averages
        hourly_avg = df.groupby(df.index.hour)[col].transform('mean')
        df[col] = df[col].fillna(hourly_avg)

        # Step 3: Fallback - if still any NaN
        df[col] = df[col].fillna(df[col].mean())

    return df

### Station Wrappers

In [6]:
from abc import ABC, abstractmethod
from typing import Dict, List, Optional
from datetime import datetime

class BaseStationWrapper(ABC):
    """
    Abstract base class for station-specific wrappers
    """

    def __init__(self, api_client: IMSWeatherAPI, station_id: int, station_name: str):
        """
        Initialize the station wrapper

        Args:
            api_client: Instance of IMSWeatherAPI
            station_id: The station ID for this wrapper
            station_name: The station name for this wrapper
        """
        self.api = api_client
        self.station_id = station_id
        self.station_name = station_name
        self._station_info = None

    def get_station_info(self) -> Dict:
        """Get and cache station information"""
        if self._station_info is None:
            self._station_info = self.api.get_station_info(self.station_id)
        return self._station_info

    def get_raw_data(self, from_date: datetime, to_date: datetime,
                     channel_id: Optional[int] = None) -> Dict:
        """
        Get raw data from the API

        Args:
            from_date: Start date
            to_date: End date
            channel_id: Optional channel ID
        """
        return self.api.get_hourly_data_by_date_range(
            station_id=self.station_id,
            from_year=from_date.year,
            from_month=from_date.month,
            from_day=from_date.day,
            to_year=to_date.year,
            to_month=to_date.month,
            to_day=to_date.day,
            channel_id=channel_id
        )

    @abstractmethod
    def get_normalized_data(self, from_date: datetime, to_date: datetime) -> List[Dict]:
        """
        Get data in a normalized format

        Returns:
            List of dictionaries with standardized keys:
            {
                'timestamp': datetime,
                'temperature': float,
                'humidity': float,
                'solar radiation': float
                'pressure': float
            }
        """
        pass

    @abstractmethod
    def _parse_station_data(self, raw_data: Dict) -> List[Dict]:
        """
        Parse station-specific raw data into normalized format

        Args:
            raw_data: Raw API response

        Returns:
            List of normalized data dictionaries
        """
        pass

class StationAshalim(BaseStationWrapper):

    def __init__(self, api_client: IMSWeatherAPI):
        super().__init__(api_client, station_id=381, station_name="Ashalim")

    def get_normalized_data(self, from_date: datetime, to_date: datetime) -> List[Dict]:
        """Get normalized data for Station 381"""
        raw_data = self.get_raw_data(from_date, to_date)
        return self._parse_station_data(raw_data)

    def _parse_station_data(self, raw_data: Dict) -> List[Dict]:

        normalized = []

        data_records = raw_data.get('data', [])

        for record in data_records:
            # Create a dictionary to easily access channel values by name
            channels_dict = {}
            for channel in record.get('channels', []):
                if channel.get('valid', False):  # Only use valid data
                    channels_dict[channel['name']] = channel['value']

            # Map the channels to standardized fields
            normalized_record = {
                'date_time': datetime.fromisoformat(record.get('datetime')),
                'temperature': channels_dict.get('TD'),  # TD = Temperature
                'humidity': channels_dict.get('RH'),  # RH = Relative Humidity
                'solar_radiation': channels_dict.get('Grad'),  # Grad = Solar Radiation
                'pressure': channels_dict.get('BP')  # BP = Pressure
            }

            normalized.append(normalized_record)

        return normalized

    def _parse_timestamp(self, timestamp_str: str) -> datetime:
        """Parse timestamp from Station 381 format"""
        # TODO: Implement based on actual timestamp format
        pass

# Usage example
if __name__ == "__main__":
    # Initialize API client
    api_client = IMSWeatherAPI(api_token="Insert IMS API token here")

    # Create station wrappers
    station_ashalim = StationAshalim(api_client)

    # Get normalized data
    from_date = datetime(2024, 12, 1)
    to_date = datetime(2025, 12, 1)

    data = station_ashalim.get_normalized_data(from_date, to_date)
    df = fill_data_gaps(data)
    df.to_csv("data.csv")



Fetching data for station 381
Date range: 2024-12-01 to 2025-12-01
Total batches: 2 (each ~6 months)

[1/2] Fetching: 2024-12-01 → 2025-06-01 ... ✓
[2/2] Fetching: 2025-06-01 → 2025-12-01 ... ✓

✓ All batches completed successfully!

